# Silver Layer — TPCH

Second phase of the medallion architecture. Reads the raw `bda_bronze` tables and produces a cleansed, conformed `bda_silver`: invalid records are quarantined with rejection reasons, `lineitem` is enriched with derived business measures, and the remaining tables pass through unchanged. Silver tables are the single source for the future gold layer.

## Setup

Create `bda_silver` if missing; refresh tables without deleting the schema.

### part — validation with quarantine

Records with an empty or NULL `p_brand` / `p_mfgr` are routed to `bda_silver.part_quarantine` with a `rejection_reason`; valid records are loaded into `bda_silver.part`, where NOT NULL and CHECK constraints enforce the same rules on all future writes.

### Lineitem — enrichment

Joins `lineitem` to `partsupp` on the part + supplier key to derive two business measures per line: `revenue` (extended price after discount) and `margin` (revenue minus supply cost of the sold quantity). Join helper columns are dropped after the enrichment.

### Remaining tables — pass-through

`customer`, `nation`, `orders`, `partsupp`, `region` and `supplier` have no additional validation rules, so they are copied from bronze to silver unchanged (including `ingestion_timestamp` for lineage). `part` and `lineitem` are intentionally excluded here — they are handled by the dedicated cells above.

In [ ]:
# Portability: target catalog and source data location are parameters.
# Defaults reproduce the original setup (Databricks Free Edition); override them via widgets or job parameters.
import re
dbutils.widgets.text("catalog", "workspace", "Target catalog")
dbutils.widgets.text("source_schema", "samples.tpch", "Source data (catalog.schema)")
CATALOG = dbutils.widgets.get("catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError(f"invalid catalog: {CATALOG!r}")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*)?", SOURCE_SCHEMA):
    raise ValueError(f"invalid source_schema: {SOURCE_SCHEMA!r}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  source_schema={SOURCE_SCHEMA}")

In [0]:
# TPC-H README for exploration only; skipped when the workspace has no /dbfs sample datasets
try:
    with open("/dbfs/databricks-datasets/tpch/README.md", "r") as f:
        content = f.read()
        displayHTML(f"<pre>{content}</pre>")
except OSError as e:
    print(f"TPC-H README not available in this workspace ({e}); continuing.")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bda_silver")

## Validation rules

Non-empty brand and manufacturer

In [0]:
from pyspark.sql.functions import col, trim, when, lit

df_part_bronze = spark.read.table("bda_bronze.part")

# Invalid records → quarantine
df_part_quarantine = df_part_bronze.filter(
    col("p_brand").isNull() |
    (trim(col("p_brand")) == "") |
    col("p_mfgr").isNull() |
    (trim(col("p_mfgr")) == "")
).withColumn(
    "rejection_reason",
    when(
        col("p_brand").isNull() | (trim(col("p_brand")) == ""),
        lit("Empty or NULL brand")
    ).when(
        col("p_mfgr").isNull() | (trim(col("p_mfgr")) == ""),
        lit("Empty or NULL manufacturer")
    )
)
df_part_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bda_silver.part_quarantine")

# Valid records → Silver
df_part_silver = df_part_bronze.filter(
    col("p_brand").isNotNull() &
    (trim(col("p_brand")) != "") &
    col("p_mfgr").isNotNull() &
    (trim(col("p_mfgr")) != "")
)

df_part_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bda_silver.part")

# Enforce constraints on Silver (drop first so the notebook can be re-run)
spark.sql("ALTER TABLE bda_silver.part DROP CONSTRAINT IF EXISTS valid_brand")
spark.sql("ALTER TABLE bda_silver.part DROP CONSTRAINT IF EXISTS valid_manufacturer")

spark.sql("""
ALTER TABLE bda_silver.part
ALTER COLUMN p_brand SET NOT NULL
""")

spark.sql("""
ALTER TABLE bda_silver.part
ALTER COLUMN p_mfgr SET NOT NULL
""")

spark.sql("""
ALTER TABLE bda_silver.part
ADD CONSTRAINT valid_brand
CHECK (TRIM(p_brand) <> '')
""")

spark.sql("""
ALTER TABLE bda_silver.part
ADD CONSTRAINT valid_manufacturer
CHECK (TRIM(p_mfgr) <> '')
""")

In [0]:
df_lineitem_bronze = spark.read.table("bda_bronze.lineitem")
df_partsupp_bronze = spark.read.table("bda_bronze.partsupp").select(
    "ps_partkey", "ps_suppkey", "ps_supplycost"
)

df_lineitem_silver = (
    df_lineitem_bronze.join(
        df_partsupp_bronze,
        (col("l_partkey") == col("ps_partkey")) &
        (col("l_suppkey") == col("ps_suppkey")),
        "left"
    )
    .withColumn(
        "revenue",
        col("l_extendedprice") * (1 - col("l_discount"))
    )
    .withColumn(
        "margin",
        col("revenue") - col("l_quantity") * col("ps_supplycost")
    )
    .drop("ps_partkey", "ps_suppkey", "ps_supplycost")
)
df_lineitem_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bda_silver.lineitem")

In [0]:
remaining_tables = [
    "customer", "nation", "orders",
    "partsupp", "region", "supplier"
]

for table in remaining_tables:
    df = spark.read.table(f"bda_bronze.{table}")
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"bda_silver.{table}")